# Análisis Exploratorio de Datos (EDA)
## Proyecto 2 - Product Development

Este notebook contiene el análisis exploratorio completo del dataset de ventas, incluyendo:
- Comprensión inicial del dataset
- Validación de la estructura de datos
- Análisis de valores faltantes y duplicados
- Análisis descriptivo general
- Exploración temporal
- Análisis comparativo entre sucursales y productos
- Visualizaciones clave
- Análisis de autocorrelación
- Insights y conclusiones

In [1]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import os

# Configuraciones
warnings.filterwarnings('ignore')
plt.style.use('default')
sns.set_palette("husl")

# Configurar tamaño de figuras por defecto
plt.rcParams['figure.figsize'] = (12, 6)

print("Librerías importadas exitosamente")

Librerías importadas exitosamente


## 1. Comprensión inicial del dataset

### a. Cargar el dataset y visualizar las primeras filas

In [3]:
# Cargar el dataset
data_path = '../data/raw/final_submission.csv'
df = pd.read_csv(data_path)

print("Dataset cargado exitosamente")
print(f"Primeras 10 filas del dataset:")
df.head(10)

Dataset cargado exitosamente
Primeras 10 filas del dataset:


,date,store,item,sales
0,2018-01-01,1,1,15.251980
1,2018-01-02,1,1,18.364534
2,2018-01-03,1,1,20.393303
3,2018-01-04,1,1,22.116090
4,2018-01-05,1,1,26.130392
5,2018-01-06,1,1,24.435510
6,2018-01-07,1,1,11.297743
7,2018-01-08,1,1,8.473736
8,2018-01-09,1,1,9.430330
9,2018-01-10,1,1,9.956382


### b. Tamaño del dataset

In [4]:
# Mostrar el tamaño del dataset
print(f"Dimensiones del dataset: {df.shape}")
print(f"Número de filas: {df.shape[0]:,}")
print(f"Número de columnas: {df.shape[1]}")
print(f"Tamaño total de datos: {df.size:,} valores")

Dimensiones del dataset: (45000, 4)
Número de filas: 45,000
Número de columnas: 4
Tamaño total de datos: 180,000 valores


### c. Descripción de las columnas

In [5]:
# Descripción de las columnas
print("Columnas del dataset:")
for col in df.columns:
    print(f"- {col}")

print("\nDescripción de cada columna:")
print("• date: Fecha de la venta (formato YYYY-MM-DD)")
print("• store: Identificador numérico de la sucursal")
print("• item: Identificador numérico del producto")
print("• sales: Cantidad de ventas registradas (variable objetivo)")

print(f"\nInformación general del dataset:")
df.info()

Columnas del dataset:
- date
- store
- item
- sales

Descripción de cada columna:
• date: Fecha de la venta (formato YYYY-MM-DD)
• store: Identificador numérico de la sucursal
• item: Identificador numérico del producto
• sales: Cantidad de ventas registradas (variable objetivo)

Información general del dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45000 entries, 0 to 44999
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   date    45000 non-null  object 
 1   store   45000 non-null  int64  
 2   item    45000 non-null  int64  
 3   sales   45000 non-null  float64
dtypes: float64(1), int64(2), object(1)
memory usage: 1.4+ MB


### d. Rango temporal de los datos

In [6]:
# Convertir la columna date a datetime
df['date'] = pd.to_datetime(df['date'])

# Identificar rango temporal
fecha_minima = df['date'].min()
fecha_maxima = df['date'].max()
duracion = fecha_maxima - fecha_minima

print(f"Rango temporal de los datos:")
print(f"• Fecha mínima: {fecha_minima.strftime('%Y-%m-%d')}")
print(f"• Fecha máxima: {fecha_maxima.strftime('%Y-%m-%d')}")
print(f"• Duración total: {duracion.days} días ({duracion.days/365.25:.1f} años)")

# Verificar continuidad temporal
fechas_unicas = df['date'].nunique()
dias_esperados = (fecha_maxima - fecha_minima).days + 1
print(f"• Fechas únicas en el dataset: {fechas_unicas}")
print(f"• Días esperados en el rango: {dias_esperados}")
print(f"• ¿Datos continuos?: {'Sí' if fechas_unicas == dias_esperados else 'No'}")

Rango temporal de los datos:
• Fecha mínima: 2018-01-01
• Fecha máxima: 2018-03-31
• Duración total: 89 días (0.2 años)
• Fechas únicas en el dataset: 90
• Días esperados en el rango: 90
• ¿Datos continuos?: Sí


## 2. Validación de la estructura de los datos

### a. Verificación de tipos de datos

In [7]:
# Verificar tipos de datos
print("Tipos de datos por columna:")
print(df.dtypes)

print("\nClasificación de variables:")
print("• date: datetime64[ns] - Variable temporal")
print("• store: int64 - Variable categórica (identificador)")
print("• item: int64 - Variable categórica (identificador)") 
print("• sales: float64 - Variable numérica continua (objetivo)")

# Verificar valores únicos en variables categóricas
print(f"\nValores únicos:")
print(f"• Número de tiendas únicas: {df['store'].nunique()}")
print(f"• Número de productos únicos: {df['item'].nunique()}")
print(f"• Rango de tiendas: {df['store'].min()} - {df['store'].max()}")
print(f"• Rango de productos: {df['item'].min()} - {df['item'].max()}")

Tipos de datos por columna:
date     datetime64[ns]
store             int64
item              int64
sales           float64
dtype: object

Clasificación de variables:
• date: datetime64[ns] - Variable temporal
• store: int64 - Variable categórica (identificador)
• item: int64 - Variable categórica (identificador)
• sales: float64 - Variable numérica continua (objetivo)

Valores únicos:
• Número de tiendas únicas: 10
• Número de productos únicos: 50
• Rango de tiendas: 1 - 10
• Rango de productos: 1 - 50


### b. Confirmación de formato datetime

In [11]:
# Confirmar formato datetime
print(f"Tipo de dato de la columna 'date': {df['date'].dtype}")
print(f"¿Es datetime?: {pd.api.types.is_datetime64_any_dtype(df['date'])}")

# Mostrar algunos ejemplos de fechas
print(f"\nEjemplos de fechas:")
print(df['date'].head(10).dt.strftime('%Y-%m-%d (%A)'))


Tipo de dato de la columna 'date': datetime64[ns]
¿Es datetime?: True

Ejemplos de fechas:
0       2018-01-01 (Monday)
1      2018-01-02 (Tuesday)
2    2018-01-03 (Wednesday)
3     2018-01-04 (Thursday)
4       2018-01-05 (Friday)
5     2018-01-06 (Saturday)
6       2018-01-07 (Sunday)
7       2018-01-08 (Monday)
8      2018-01-09 (Tuesday)
9    2018-01-10 (Wednesday)
Name: date, dtype: object


### c. Validación de frecuencia temporal

In [12]:
print(f"\n### c. Validación de frecuencia temporal")

# Analizar diferencias entre fechas consecutivas
df_sorted = df.sort_values('date')
fechas_unicas_sorted = df_sorted['date'].drop_duplicates().sort_values()
diferencias = fechas_unicas_sorted.diff().dropna()

print(f"Diferencias entre fechas consecutivas:")
print(f"• Diferencia más común: {diferencias.mode().iloc[0]}")
print(f"• Diferencias únicas: {diferencias.unique()}")

# Verificar si es diaria
es_diaria = all(diferencias == pd.Timedelta(days=1))
print(f"• ¿Frecuencia diaria?: {'Sí' if es_diaria else 'No'}")

if es_diaria:
    print("✓ Los datos tienen frecuencia temporal diaria consistente")
else:
    print("⚠ Los datos no tienen frecuencia diaria consistente")
    print(f"  Distribución de diferencias: {diferencias.value_counts()}")


### c. Validación de frecuencia temporal
Diferencias entre fechas consecutivas:
• Diferencia más común: 1 days 00:00:00
• Diferencias únicas: <TimedeltaArray>
['1 days']
Length: 1, dtype: timedelta64[ns]
• ¿Frecuencia diaria?: Sí
✓ Los datos tienen frecuencia temporal diaria consistente


## 3. Análisis de valores faltantes y duplicados

### a. Identificación de valores nulos

In [9]:
# Identificar valores nulos
valores_nulos = df.isnull().sum()
porcentaje_nulos = (df.isnull().sum() / len(df)) * 100

print("Análisis de valores faltantes:")
print("=" * 40)
for col in df.columns:
    print(f"{col:8}: {valores_nulos[col]:8} nulos ({porcentaje_nulos[col]:5.2f}%)")

print(f"\nTotal de filas: {len(df):,}")
print(f"Total de valores nulos: {valores_nulos.sum():,}")
print(f"Porcentaje total de valores nulos: {(valores_nulos.sum() / df.size) * 100:.2f}%")

# Visualización de valores nulos si existen
if valores_nulos.sum() > 0:
    plt.figure(figsize=(10, 6))
    valores_nulos.plot(kind='bar')
    plt.title('Valores Nulos por Columna')
    plt.ylabel('Cantidad de Valores Nulos')
    plt.xticks(rotation=45)
    plt.show()
else:
    print("\n✓ No se encontraron valores nulos en el dataset")

Análisis de valores faltantes:
date    :        0 nulos ( 0.00%)
store   :        0 nulos ( 0.00%)
item    :        0 nulos ( 0.00%)
sales   :        0 nulos ( 0.00%)

Total de filas: 45,000
Total de valores nulos: 0
Porcentaje total de valores nulos: 0.00%

✓ No se encontraron valores nulos en el dataset


### b. Estrategia para valores faltantes y análisis de duplicados

In [10]:
# Estrategia para valores faltantes
print("ESTRATEGIA PARA VALORES FALTANTES:")
print("=" * 50)
if valores_nulos.sum() == 0:
    print("✓ No se requiere tratamiento de valores faltantes")
    print("  El dataset está completo sin valores nulos")
else:
    print("⚠ Se detectaron valores faltantes:")
    for col in df.columns[valores_nulos > 0]:
        print(f"  - {col}: {valores_nulos[col]} valores ({porcentaje_nulos[col]:.2f}%)")
    print("\nEstrategia recomendada:")
    print("  1. Si < 5%: Eliminar filas o imputar con mediana/moda")
    print("  2. Si 5-20%: Imputación avanzada (interpolación temporal)")
    print("  3. Si > 20%: Investigar patrones y considerar variables auxiliares")

print("\n" + "="*50)
print("ANÁLISIS DE REGISTROS DUPLICADOS:")
print("=" * 50)

# Verificar duplicados completos
duplicados_completos = df.duplicated().sum()
print(f"Registros completamente duplicados: {duplicados_completos}")

# Verificar duplicados por combinación (date, store, item)
duplicados_clave = df.duplicated(subset=['date', 'store', 'item']).sum()
print(f"Registros con fecha-tienda-producto duplicados: {duplicados_clave}")

# Mostrar ejemplos si existen duplicados
if duplicados_clave > 0:
    print("\nEjemplos de registros duplicados:")
    duplicated_mask = df.duplicated(subset=['date', 'store', 'item'], keep=False)
    ejemplos_duplicados = df[duplicated_mask].sort_values(['date', 'store', 'item']).head(10)
    print(ejemplos_duplicados)
    
    print("\nESTRATEGIA PARA DUPLICADOS:")
    print("1. Verificar si son errores de carga o registros legítimos")
    print("2. Si son errores: eliminar duplicados manteniendo el primer registro")
    print("3. Si son legítimos: sumar las ventas para la misma fecha-tienda-producto")
else:
    print("✓ No se encontraron registros duplicados")
    print("  Cada combinación fecha-tienda-producto es única")

ESTRATEGIA PARA VALORES FALTANTES:
✓ No se requiere tratamiento de valores faltantes
  El dataset está completo sin valores nulos

ANÁLISIS DE REGISTROS DUPLICADOS:
Registros completamente duplicados: 0
Registros con fecha-tienda-producto duplicados: 0
✓ No se encontraron registros duplicados
  Cada combinación fecha-tienda-producto es única
